### Data Fetching

In [19]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_14624\1965381518.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)


✅ Fetched 7760 rows from 'extraction'


In [20]:
keywords = ["CCTV"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df

,filename,workorder_id,json_data
2,SC_PM_NA_CCTV_NA_19.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
218,SC_PM_NA_CCTV_NA_20 (2).pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
738,SC_PM_NA_CCTV_NA_2 (2).pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
776,SC_PM_NA_CCTV_NA_23 (1).pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
809,SC_PM_QTR_CCTV_4000480655.pdf,4.000481e+09,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...
6414,SC_PM_NA_CCTV_NA_16 (3).pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
6489,SC_PM_NA_CCTV_NA_18 (2).pdf,NaN,{'notification': {'notification_no': '12165934...
6497,SC_PM_NA_CCTV_NA_18.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
6518,SC_PM_NA_CCTV_NA_19 (2).pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."


In [21]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['cctv', 'cctv_status', 'comment_recommendation', 'notification', 'performed_by', 'reference_document', 'verified_by', 'work_order']


In [22]:
updated_json_list = []

for item in valid_json:
    item.pop('notification', None)
    item.pop('work_order', None)

    if 'cctv' not in item or not isinstance(item['cctv'], dict):
        item['cctv'] = {}

    for key in ['cctv_status', 'comment_recommendation', 'performed_by', 'verified_by']:
        if key in item:
            item['cctv'][key] = item.pop(key)

    updated_json_list.append(item)

df_cctv = df.loc[valid_json.index, ['workorder_id', 'filename']].copy()
df_cctv['json_data'] = updated_json_list

df_cctv

,workorder_id,filename,json_data
2,NaN,SC_PM_NA_CCTV_NA_19.pdf,"{'cctv': {'station': 'MKU', 'pm_order': 'NA', ..."
218,NaN,SC_PM_NA_CCTV_NA_20 (2).pdf,"{'cctv': {'station': 'MAH', 'pm_order': 'NA', ..."
738,NaN,SC_PM_NA_CCTV_NA_2 (2).pdf,"{'cctv': {'station': 'MLU', 'pm_order': 'NA', ..."
776,NaN,SC_PM_NA_CCTV_NA_23 (1).pdf,"{'cctv': {'station': 'KLS', 'pm_order': 'NA', ..."
809,4.000481e+09,SC_PM_QTR_CCTV_4000480655.pdf,"{'cctv': {'station': 'BAS', 'pm_order': 'NA', ..."
...,...,...,...
6414,NaN,SC_PM_NA_CCTV_NA_16 (3).pdf,"{'cctv': {'station': 'HAH', 'pm_order': 'NA', ..."
6489,NaN,SC_PM_NA_CCTV_NA_18 (2).pdf,"{'cctv': {'station': 'Hah', 'pm_order': 'NA', ..."
6497,NaN,SC_PM_NA_CCTV_NA_18.pdf,"{'cctv': {'station': 'HAH', 'pm_order': 'NA', ..."
6518,NaN,SC_PM_NA_CCTV_NA_19 (2).pdf,"{'cctv': {'station': 'HAH', 'pm_order': 'NA', ..."


In [23]:
import pandas as pd
import json

valid_mask = df_cctv['json_data'].apply(lambda x: isinstance(x, dict))

rows = []

for _, row in df_cctv[valid_mask].iterrows():
    base_data = {
        'workorder_id': row['workorder_id'],
        'filename': row['filename']
    }

    cctv_data = row['json_data'].get('cctv', {})

    for key, value in cctv_data.items():
        if isinstance(value, dict):
            base_data[key] = json.dumps(value)
        else:
            base_data[key] = value

    rows.append(base_data)

df_cctv = pd.DataFrame(rows)

df_cctv = df_cctv.drop(
    columns=[col for col in df_cctv.columns if col.startswith("reference")],
    errors="ignore"
)

df_cctv

,workorder_id,filename,station,pm_order,datetime,procedures,cctv_status,comment_recommendation,performed_by,verified_by,remark
0,NaN,SC_PM_NA_CCTV_NA_19.pdf,MKU,NA,22/02/2022,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": true, ""alignment""...",X - need to continue checking,7223,7111,NaN
1,NaN,SC_PM_NA_CCTV_NA_20 (2).pdf,MAH,NA,16/09/2023,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": true, ""alignment""...",Good Condition,11727,7111,NaN
2,NaN,SC_PM_NA_CCTV_NA_2 (2).pdf,MLU,NA,22/05/2023,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": true, ""alignment""...",all equipment in good conditions,11727,7111,NaN
3,NaN,SC_PM_NA_CCTV_NA_23 (1).pdf,KLS,NA,14/06/2023,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": ""NA"", ""alignment""...",NA,7222,7111,NaN
4,4.000481e+09,SC_PM_QTR_CCTV_4000480655.pdf,BAS,NA,21/08/2022,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": ""ok"", ""alignment""...","all equipment in good conditions, x= need cont...",11727,7111,NaN
...,...,...,...,...,...,...,...,...,...,...,...
319,NaN,SC_PM_NA_CCTV_NA_16 (3).pdf,HAH,NA,17/03/2024,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": ""ok"", ""alignment""...",All in a good conditions.,7262,7111,NaN
320,NaN,SC_PM_NA_CCTV_NA_18 (2).pdf,Hah,NA,17/08/2023,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": ""ok"", ""alignment""...",good condition,7248,7111,NaN
321,NaN,SC_PM_NA_CCTV_NA_18.pdf,HAH,NA,18/06/2022,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": ""ok"", ""alignment""...",all equipment in good conditions no need to co...,11727,7111,NaN
322,NaN,SC_PM_NA_CCTV_NA_19 (2).pdf,HAH,NA,17/09/2023,[{'safety_and_preparation': {'a': {'procedure'...,"{""CA01"": {""picture_quality"": ""ok"", ""alignment""...",Good Condition,11727,7111,NaN


In [24]:
df_cctv.columns

Index(['workorder_id', 'filename', 'station', 'pm_order', 'datetime',
       'procedures', 'cctv_status', 'comment_recommendation', 'performed_by',
       'verified_by', 'remark'],
      dtype='object')

In [25]:
def flatten_procedures(proc_list):
    """Flattens the nested list of dictionaries in 'procedures'."""
    flat_data = {}
    if not isinstance(proc_list, list): return flat_data
    
    def walk_dict(d, parent_key=''):
        for k, v in d.items():
            new_key = f"{parent_key}.{k}" if parent_key else k
            if isinstance(v, dict):
                if 'status' in v:
                    flat_data[f"{new_key}.status"] = v['status']
                else:
                    walk_dict(v, new_key)
    
    for item in proc_list:
        walk_dict(item)
    return flat_data

def flatten_cctv_status(status_data):
    """Flattens the camera dictionary, handling strings/dicts and converting NA to N/A."""
    flat_data = {}
    
    # 1. Handle string inputs (convert JSON string to dict)
    if isinstance(status_data, str) and status_data.strip():
        try:
            # Replacing single quotes with double quotes for valid JSON
            status_data = json.loads(status_data.replace("'", '"'))
        except (json.JSONDecodeError, AttributeError):
            return flat_data
            
    # 2. Validation: Ensure we are working with a dictionary
    if not isinstance(status_data, dict): 
        return flat_data
    
    # 3. Flattening logic
    for camera_id, attributes in status_data.items():
        if isinstance(attributes, dict):
            for attr_name, attr_value in attributes.items():
                # Check for NA values: None, empty strings, or "NA"
                if attr_value is None or attr_value == "" or str(attr_value).upper() == "NA":
                    clean_value = "N/A"
                else:
                    clean_value = attr_value
                
                flat_data[f"{camera_id}.{attr_name}"] = clean_value
                
    return flat_data

df_proc_flat = df_cctv["procedures"].apply(flatten_procedures).apply(pd.Series)
df_status_flat = df_cctv["cctv_status"].apply(flatten_cctv_status).apply(pd.Series)

# 3. Combine EVERYTHING
# We take the original metadata, then add the procedure columns, then the camera columns
df_final = pd.concat([
    df_cctv.drop(columns=['procedures', 'cctv_status', 'pm_order']), 
    df_proc_flat, 
    df_status_flat
], axis=1)

# Display the result
df_final.columns

Index(['workorder_id', 'filename', 'station', 'datetime',
       'comment_recommendation', 'performed_by', 'verified_by', 'remark',
       'safety_and_preparation.a.status', 'safety_and_preparation.b.status',
       ...
       'CAMO4.picture_quality', 'CAMO4.alignment', 'CAMO5.picture_quality',
       'CAMO5.alignment', 'CAM15.picture_quality', 'CAM15.alignment',
       'CAM16.picture_quality', 'CAM16.alignment', 'PT02.picture_quality',
       'PT02.alignment'],
      dtype='object', length=178)

In [26]:
for i in df_final.columns:
    print(i)

workorder_id
filename
station
datetime
comment_recommendation
performed_by
verified_by
remark
safety_and_preparation.a.status
safety_and_preparation.b.status
cleanliness.a.status
cleanliness.b.status
cleanliness.c.status
cleanliness.d.status
cleanliness.e.status
cables.a.status
cables.b.status
cctv_workstation_healthiness_check.a.status
cctv_workstation_healthiness_check.b.status
dvr.cleanliness_of_the_dvr.a.status
dvr.cleanliness_of_the_dvr.b.status
dvr.cables.c.status
dvr.cables.d.status
cleanliness_and_function.a.status
cleanliness_and_function.b.status
ptz_cameras.a.status
ptz_cameras.b.status
see_eyes.a.status
see_eyes.b.status
see_eyes.c.status
CA01.picture_quality
CA01.alignment
CA02.picture_quality
CA02.alignment
CA03.picture_quality
CA03.alignment
CA04.picture_quality
CA04.alignment
CA05.picture_quality
CA05.alignment
CA06.picture_quality
CA06.alignment
CA07.picture_quality
CA07.alignment
CA08.picture_quality
CA08.alignment
CA09.picture_quality
CA09.alignment
CA10.picture_qual

In [27]:
output_file = f"../../output/snc/cctv.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='cctv', index=False)

print(f"Saved excel to {output_file}")

Saved excel to ../../output/snc/cctv.xlsx
